In [ ]:
# LIBRERIE

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import KNNImputer
from sklearn.model_selection import train_test_split

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor

from missforest import MissForest
from sklearn.preprocessing import OrdinalEncoder

In [ ]:
df = pd.read_csv("cmi_internet.csv")
legenda = pd.read_csv("data_dictionary.csv")

In [ ]:
df_da_imp = df.copy()

# Creazione variabili utili

# Tempo
df_da_imp['Fitness_Endurance-Tempo_totale'] = df_da_imp['Fitness_Endurance-Time_Sec'] + \
    (df_da_imp['Fitness_Endurance-Time_Mins']*60)

# BMI dopo imputazione!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

# Drop Variabili
df_da_imp = df_da_imp.drop(["BIA-BIA_BMI", "Physical-BMI", "Fitness_Endurance-Time_Mins", "Fitness_Endurance-Time_Sec",
                            "PAQ_A-PAQ_A_Total", "PAQ_C-PAQ_C_Total", "PAQ_A-Season", "PAQ_C-Season"], axis=1)

df_da_imp = df_da_imp[df["Basic_Demos-Age"] <= 18]

colonne_zeri = ['Physical-Weight',
                'Physical-Diastolic_BP', 'Physical-Systolic_BP']
df_da_imp[colonne_zeri] = df_da_imp[colonne_zeri].mask(
    lambda x: x == 0, np.nan)

# Fuori Scala
# CGAS-CGAS_Score
df_da_imp["CGAS-CGAS_Score"] = df_da_imp["CGAS-CGAS_Score"].mask(
    lambda x: x > 100, np.nan)

In [ ]:
'''# Colonne just to check
col_categoriche = df.select_dtypes(include=["object"]).columns

categoriche = df[col_categoriche]
stagioni = {"Winter": 0, "Spring": 1, "Summer": 2, "Fall": 3}
tutto_numerico = df.copy()
tutto_numerico[col_categoriche] = tutto_numerico[col_categoriche].replace(
    stagioni)
correlazione = tutto_numerico.corr()

col_numeriche = df.select_dtypes(
    include=["int64", "float64"]).columns
col_quantitative = df.select_dtypes(
    include=["int64", "float64"]).columns.drop("id")

col_da_escludere = [
    "FGC-FGC_CU_Zone", "FGC-FGC_GSND_Zone", "FGC-FGC_GSD_Zone",
    "FGC-FGC_PU_Zone", "FGC-FGC_SRL_Zone", "FGC-FGC_SRR_Zone",
    "FGC-FGC_TL_Zone"
]

col_non_bin = col_quantitative.difference(col_da_escludere)'''

In [ ]:
'''sns.set_theme(style="white")


def plot_professional_matrix(corr_matrix, title="Correlation Matrix"):
    # Creazione di una singola figura
    fig, ax = plt.subplots(figsize=(10, 8), constrained_layout=True)

    # Maschera per nascondere la parte superiore (triangolo)
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

    # Palette divergente professionale (es. Rosso-Blu)
    cmap = sns.diverging_palette(230, 20, as_cmap=True)

    # Generazione della heatmap
    sns.heatmap(corr_matrix,
                mask=mask,
                cmap=cmap,
                vmax=1, vmin=-1, center=0,
                square=True,
                linewidths=.5,
                cbar_kws={"shrink": .5},
                annot=False,             # Impostare a True per mostrare i numeri
                fmt=".2f",               # Formato arrotondamento decimali
                ax=ax)

    # Impostazione del titolo
    ax.set_title(title, fontsize=16, pad=20)

    plt.show()


plot_professional_matrix(correlazione)'''

In [ ]:
# Split DF
train_set, test_set = train_test_split(
    df_da_imp, test_size=0.3, stratify=df_da_imp["sii"], random_state=222)

In [ ]:
# imputazione
# Ci ho dormito sopra, usare Knn (nonostante lo abbia detto l'altro Riccardo) è sbagliato su più punti, va rifatta l'imputazione con Random Forest o simili
# Intanto non si è fatto feature scaling prima del calcolo delle distanze
# Poi, se imputiamo dati mancanti di un'unità kinda vicina ad outliers ce la possibilità che si vada a creare un cluster, ed i metodi a densità non li identificano come tali
# Infine ce anche un discorso di facile propagazione di valori outliers, che potrebbero influenzare u po' troppo l'intero dataset

'''imputer = KNNImputer(n_neighbors=76)  # np.sqrt(5823)
train_imputed = pd.DataFrame(
    imputer.fit_transform(train_set[col_non_bin]),
    columns=col_non_bin,
    index=train_set.index)

test_imputed = pd.DataFrame(
    imputer.transform(test_set[col_non_bin]),
    columns=col_non_bin,
    index=test_set.index)'''

In [ ]:
# Imputazione

rf_estimator = RandomForestRegressor(random_state=222, n_jobs=-1)
imputer_rf = IterativeImputer(
    estimator=rf_estimator, random_state=222, max_iter=5)

train_imputed_rf = pd.DataFrame(
    imputer_rf.fit_transform(train_set[col_non_bin]),
    columns=col_non_bin,
    index=train_set.index
)

test_imputed_rf = pd.DataFrame(
    imputer_rf.transform(test_set[col_non_bin]),
    columns=col_non_bin,
    index=test_set.index
)

In [ ]:
train_set.shape

In [ ]:
from missforest import MissForest
from sklearn.preprocessing import OrdinalEncoder

variabili_cat = ['Basic_Demos-Enroll_Season', 'Basic_Demos-Sex', 'CGAS-Season',
                 'Physical-Season', 'Fitness_Endurance-Season', 'FGC-Season',
                 "FGC-FGC_CU_Zone", "FGC-FGC_GSND_Zone", "FGC-FGC_GSD_Zone",
                 "FGC-FGC_PU_Zone", "FGC-FGC_SRL_Zone", "FGC-FGC_SRR_Zone",
                 "FGC-FGC_TL_Zone", 'BIA-Season', 'BIA-BIA_Activity_Level_num', 'BIA-BIA_Frame_num',
                 'PCIAT-Season', 'PCIAT-PCIAT_01', 'PCIAT-PCIAT_02', 'PCIAT-PCIAT_03',
                 'PCIAT-PCIAT_04', 'PCIAT-PCIAT_05', 'PCIAT-PCIAT_06', 'PCIAT-PCIAT_07',
                 'PCIAT-PCIAT_08', 'PCIAT-PCIAT_09', 'PCIAT-PCIAT_10', 'PCIAT-PCIAT_11',
                 'PCIAT-PCIAT_12', 'PCIAT-PCIAT_13', 'PCIAT-PCIAT_14', 'PCIAT-PCIAT_15',
                 'PCIAT-PCIAT_16', 'PCIAT-PCIAT_17', 'PCIAT-PCIAT_18', 'PCIAT-PCIAT_19',
                 'PCIAT-PCIAT_20', 'SDS-Season', 'PreInt_EduHx-Season',
                 'PreInt_EduHx-computerinternet_hoursday']

# Isolamento delle colonne di tipo object per l'encoding
object_cols = train_set.select_dtypes(include=['object']).columns

# Inizializzazione dell'encoder (istruito per ignorare i NaN durante il fit)
# Nelle versioni recenti di scikit-learn, encoded_missing_value gestisce i NaN
encoder = OrdinalEncoder(handle_unknown='use_encoded_value',
                         unknown_value=-1, encoded_missing_value=np.nan)

# Trasformazione: i NaN rimangono NaN (per permettere a MissForest di individuarli)
train_set_encoded = train_set.copy()
train_set_encoded[object_cols] = encoder.fit_transform(
    train_set_encoded[object_cols])

# Inizializzazione e imputazione
imputer_mf = MissForest(categorical=variabili_cat)
train_imputed_array = imputer_mf.fit_transform(x=train_set_encoded)

# Ricostruzione del DataFrame
train_imputed_mf = pd.DataFrame(
    train_imputed_array,
    columns=train_set.columns,
    index=train_set.index
)

# Ripristino delle etichette testuali originali sulle colonne categoriche
# (I valori imputati originariamente NaN ora avranno un'etichetta testuale)
train_imputed_mf[object_cols] = encoder.inverse_transform(
    train_imputed_mf[object_cols])

In [ ]:
# Salvataggio dati in CSV

train_imputed_rf.to_csv("train_imputed.csv", index=True)
test_imputed_rf.to_csv("test_imputed.csv", index=True)